# 02 - Silver Transformation

Standardize data types, remove duplicate business keys, clean text values, and reject invalid facts.


In [ ]:
from pyspark.sql import functions as F

customers = spark.table("bronze_customers")
products = spark.table("bronze_products")
sales = spark.table("bronze_sales")

silver_customers = (customers.select(
    F.trim("customer_id").alias("customer_id"),
    F.initcap(F.trim("customer_name")).alias("customer_name"),
    F.trim("segment").alias("segment"),
    F.initcap(F.trim("city")).alias("city"),
    F.initcap(F.trim("state")).alias("state"),
    F.trim("region").alias("region"),
    F.to_date("join_date").alias("join_date"))
    .dropDuplicates(["customer_id"]).filter(F.col("customer_id").isNotNull()))

silver_products = (products.select(
    F.trim("product_id").alias("product_id"),
    F.trim("product_name").alias("product_name"),
    F.trim("category").alias("category"),
    F.col("unit_cost").cast("decimal(18,2)").alias("unit_cost"),
    F.col("list_price").cast("decimal(18,2)").alias("list_price"))
    .dropDuplicates(["product_id"])
    .filter((F.col("product_id").isNotNull()) & (F.col("list_price") >= 0)))

silver_sales = (sales.select(
    F.trim("order_id").alias("order_id"),
    F.to_date("order_date").alias("order_date"),
    F.trim("customer_id").alias("customer_id"),
    F.trim("product_id").alias("product_id"),
    F.col("quantity").cast("int").alias("quantity"),
    F.col("unit_price").cast("decimal(18,2)").alias("unit_price"),
    F.col("discount_pct").cast("decimal(5,2)").alias("discount_pct"),
    F.col("sales_amount").cast("decimal(18,2)").alias("sales_amount"),
    F.col("cost_amount").cast("decimal(18,2)").alias("cost_amount"))
    .dropDuplicates(["order_id"])
    .filter((F.col("order_date").isNotNull()) & (F.col("quantity") > 0) &
            (F.col("sales_amount") >= 0)))

for name, df in {
    "silver_customers": silver_customers,
    "silver_products": silver_products,
    "silver_sales": silver_sales
}.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(name)
    print(name, df.count())
